## Continent pi chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import pycountry
import pycountry_convert as pc
import seaborn as sns

# ---------------------------
# 1. Load CSV files
# ---------------------------
# csv1 = pd.read_csv("file1.csv")
# csv2 = pd.read_csv("file2.csv")

# ---------------------------
# 2. Join the CSV files
# (change the column name if needed) unique_country
# ---------------------------
# df = pd.merge(csv1, csv2, on="country_code", how="inner")

# # If you just want to stack rows instead:
# df = pd.concat([csv1, csv2], ignore_index=True)


csv_file = "/home/fahimul/Documents/Research/Proj_worldDataset/csv/subset_all_final.csv"

# ======== Example Setup ========
# Read your CSV (or create DataFrame)
df = pd.read_csv(csv_file)

# -----------------------------
# Convert country code → continent
# -----------------------------
def get_continent(code):
    try:
        continent_code = pc.country_alpha2_to_continent_code(code)
        continent_map = {
            "AF": "Africa",
            "AS": "Asia",
            "EU": "Europe",
            "NA": "North America",
            "SA": "South America",
            "OC": "Oceania",
            "AN": "Antarctica"
        }
        return continent_map.get(continent_code)
    except:
        return None

df["continent"] = df["unique_country"].apply(get_continent)

# -----------------------------
# Remove Unknown / None values
# -----------------------------
df = df.dropna(subset=["continent"])

# -----------------------------
# Continent distribution
# -----------------------------
continent_counts = df["continent"].value_counts()

# -----------------------------
# Seaborn styling
# -----------------------------
sns.set_theme(style="whitegrid")
colors = sns.color_palette("pastel", len(continent_counts))

# -----------------------------
# Pie Chart
# -----------------------------
plt.figure(figsize=(8,8))

plt.pie(
    continent_counts,
    labels=continent_counts.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=colors
)

plt.title("Continent Distribution")
plt.axis("equal")

plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pycountry_convert as pc

# ---------------------------
# 1. Load CSV files
# ---------------------------
# csv1 = pd.read_csv("file1.csv")
# csv2 = pd.read_csv("file2.csv")

# ---------------------------
# 2. Join the CSV files
# (change the column name if needed) unique_country
# ---------------------------
# df = pd.merge(csv1, csv2, on="country_code", how="inner")

# # If you just want to stack rows instead:
# df = pd.concat([csv1, csv2], ignore_index=True)


csv_file = "/home/fahimul/Documents/Research/Proj_worldDataset/csv/subset_all_final.csv"

# ======== Example Setup ========
# Read your CSV (or create DataFrame)
df = pd.read_csv(csv_file) #unique_country

# Remove Brazil (country code BR)
# -----------------------------
# df = df[df["unique_country"] != "BR"]
# df = df[df["unique_country"] != "AR"]


# -----------------------------
# Downsample specific countries
# -----------------------------
def reduce_country(df, code, remove_percent):
    country_df = df[df["unique_country"] == code]
    keep_fraction = 1 - remove_percent
    kept = country_df.sample(frac=keep_fraction, random_state=42)
    df_other = df[df["unique_country"] != code]
    return pd.concat([df_other, kept])

# remove 80% of Brazil
df = reduce_country(df, "BR", 0.90)

# remove 40% of Argentina
df = reduce_country(df, "AR", 0.90)

# remove 40% of Columbia
df = reduce_country(df, "CO", 0.90)

print(df.shape)

# -----------------------------
# Convert country code → continent
# -----------------------------
def get_continent(code):
    try:
        continent_code = pc.country_alpha2_to_continent_code(code)
        continent_map = {
            "AF": "Africa",
            "AS": "Asia",
            "EU": "Europe",
            "NA": "North America",
            "SA": "South America",
            "OC": "Oceania",
            "AN": "Antarctica"
        }
        return continent_map.get(continent_code)
    except:
        return None

df["continent"] = df["unique_country"].apply(get_continent)

# -----------------------------
# Remove Unknown continents
# -----------------------------
df = df.dropna(subset=["continent"])

# -----------------------------
# Continent distribution
# -----------------------------
continent_counts = df["continent"].value_counts()

# -----------------------------
# Seaborn style
# -----------------------------
sns.set_theme(style="whitegrid")
colors = sns.color_palette("pastel", len(continent_counts))

# -----------------------------
# Pie chart
# -----------------------------
plt.figure(figsize=(8,8))

# plt.pie(
#     continent_counts,
#     labels=continent_counts.index,
#     autopct="%1.1f%%",
#     startangle=140,
#     colors=colors
# )

wedges, texts, autotexts = plt.pie(
    continent_counts,
    labels=continent_counts.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=colors
)

for text in texts:
    text.set_fontsize(20)

for autotext in autotexts:
    autotext.set_fontsize(20)


# plt.title("Continent Distribution")
plt.axis("equal")

plt.show()

## Coastal vs Inland analysis

In [ ]:
import pandas as pd
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
from shapely.geometry import Point

# -----------------------------
# Load dataset
# -----------------------------

csv_file = "/home/fahimul/Documents/Research/Proj_worldDataset/csv/subset_all_final.csv"

# ======== Example Setup ========
# Read your CSV (or create DataFrame)
df = pd.read_csv(csv_file) #unique_country


# -----------------------------
# Downsample specific countries
# -----------------------------
def reduce_country(df, code, remove_percent):
    country_df = df[df["unique_country"] == code]
    keep_fraction = 1 - remove_percent
    kept = country_df.sample(frac=keep_fraction, random_state=42)
    df_other = df[df["unique_country"] != code]
    return pd.concat([df_other, kept])

# remove 80% of Brazil
df = reduce_country(df, "BR", 0.90)

# remove 40% of Argentina
df = reduce_country(df, "AR", 0.90)

# remove 40% of Columbia
df = reduce_country(df, "CO", 0.90)

# -----------------------------
# -----------------------------

# dataset must contain latitude & longitude
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
)

# -----------------------------
# Load coastline shapefile
# -----------------------------
coastline = gpd.read_file("metadata/ne_10m_coastline/ne_10m_coastline.shp")

# Convert projection for distance calculation
gdf = gdf.to_crs(epsg=3857)
coastline = coastline.to_crs(epsg=3857)

# -----------------------------
# Compute distance to coastline
# -----------------------------
gdf["distance_to_coast_km"] = gdf.geometry.apply(
    lambda x: coastline.distance(x).min() / 1000
)

# -----------------------------
# Coastal vs Inland
# -----------------------------
threshold = 50  # km
gdf["location_type"] = gdf["distance_to_coast_km"].apply(
    lambda x: "Coastal" if x <= threshold else "Inland"
)

# -----------------------------
# Distribution
# -----------------------------
counts = gdf["location_type"].value_counts()

# -----------------------------
# Plot
# -----------------------------
sns.set_theme(style="whitegrid", font_scale=1.5)

plt.figure(figsize=(7,7))
plt.pie(counts, labels=counts.index, autopct="%1.1f%%")
plt.title("Coastal vs Inland Distribution")
plt.axis("equal")

plt.show()

## Rural vs Urban distribution

In [ ]:
import pandas as pd
import osmnx as ox
import seaborn as sns
import matplotlib.pyplot as plt


# -----------------------------
# Load dataset
# -----------------------------

csv_file = "/home/fahimul/Documents/Research/Proj_worldDataset/csv/subset_all_final.csv"

# ======== Example Setup ========
# Read your CSV (or create DataFrame)
df = pd.read_csv(csv_file) #unique_country


# -----------------------------
# Downsample specific countries
# -----------------------------
def reduce_country(df, code, remove_percent):
    country_df = df[df["unique_country"] == code]
    keep_fraction = 1 - remove_percent
    kept = country_df.sample(frac=keep_fraction, random_state=42)
    df_other = df[df["unique_country"] != code]
    return pd.concat([df_other, kept])

# remove 80% of Brazil
df = reduce_country(df, "BR", 0.90)

# remove 40% of Argentina
df = reduce_country(df, "AR", 0.90)

# remove 40% of Columbia
df = reduce_country(df, "CO", 0.90)

# -----------------------------
# -----------------------------

def classify_area(lat, lon):

    try:
        tags = {"landuse": True}
        gdf = ox.features_from_point((lat, lon), tags=tags, dist=1000)

        urban_tags = ["residential", "commercial", "industrial"]

        if any(gdf["landuse"].isin(urban_tags)):
            return "Urban"
        else:
            return "Rural"

    except:
        return "Unknown"


df["area_type"] = df.apply(
    lambda row: classify_area(row["latitude"], row["longitude"]),
    axis=1
)

# remove unknown
df = df[df["area_type"] != "Unknown"]

counts = df["area_type"].value_counts()

# plot
sns.set_theme(style="whitegrid", font_scale=1.5)

plt.figure(figsize=(7,7))

plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%"
)

plt.title("Urban vs Rural Distribution")
plt.axis("equal")

plt.show()

# Urban 80.7, rural 19.3

## Countrywise line graph retrieval scores.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# load data
df = pd.read_csv("csv/countrywise_retriv.csv")

# reshape dataframe for seaborn
df_long = df.melt(
    id_vars="country",
    value_vars=["Top 1", "Top 5", "Top 10"],
    var_name="metric",
    value_name="score"
)

# metric order
metric_order = ["Top 1", "Top 5", "Top 10"]
df_long["metric"] = pd.Categorical(df_long["metric"], metric_order)

# plotting
sns.set_theme(style="whitegrid", font_scale=1.5)

plt.figure(figsize=(10,6))

sns.lineplot(
    data=df_long,
    x="metric",
    y="score",
    hue="country",
    marker="o"
)

plt.xlabel("Retrieval Metric")
plt.ylabel("Accuracy (%)")
# plt.title("Retrieval Performance Across Countries")

plt.legend(title="Country", bbox_to_anchor=(1.05,1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
data = {
    "": ["R@1","R@5","R@10","R@1%"]*3,
    "Score": [15.19, 42.10, 54.04, 90.87,
            11.36, 37.11, 51.33, 89.40,
              3.27, 10.15, 15.48, 65.04,],
    "Ablations": ["GeoQueryNet"]*4 +["Without LoRA"]*4 + ["Without Cross-View Alignment Module"]*4
}
df = pd.DataFrame(data)

plt.figure(figsize=(10,5))
sns.barplot(x="", y="Score", hue="Ablations", data=df, palette="Set2")

# plt.xlabel("Retrieval Metrics", fontsize=14)
plt.ylabel("Score (%)", fontsize=14)
# plt.title("Performance Comparison on CVW500k", fontsize=16)

plt.legend(title="Method")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

data = {
    "": ["R@1", "R@5", "R@10", "R@1%"],
    "Score": [11.36, 37.11, 51.33, 89.40]
}

df = pd.DataFrame(data)

sns.set_style("whitegrid")

plt.figure(figsize=(8,5))
sns.barplot(x="", y="Score", data=df)

# plt.xlabel("Retrieval Metrics", fontsize=14)
plt.ylabel("Score (%)", fontsize=14)
# plt.title("GeoQueryNet Performance on CVW500k", fontsize=16)

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Example data
data = {
    "Metric": ["R@1","R@5","R@10","R@1%"] * 2,
    "Score": [4.80,13.35,18.97,69.87, 15.19,42.10,54.04,90.87],
    "Method": ["ConGeo"]*4 + ["GeoQueryNet"]*4
}

df = pd.DataFrame(data)

sns.set_style("whitegrid")

plt.figure(figsize=(8,5))

sns.barplot(
    x="Metric",
    y="Score",
    hue="Method",      # This creates side-by-side bars
    data=df,
    palette="Set2"
)

plt.xlabel("Retrieval Metrics", fontsize=14)
plt.ylabel("Score (%)", fontsize=14)
plt.title("Performance Comparison on CVW500k", fontsize=16)

plt.legend(title="Method")
plt.tight_layout()

plt.show()

# Examples of dataset


In [ ]:
# import pandas as pd
# import random
# import os
# import matplotlib.pyplot as plt
# from PIL import Image

# def show_pairs(csv_file, data_folder, country_name, num_samples):

#     # read csv
#     df = pd.read_csv(csv_file)

#     # filter by country
#     df = df[df["unique_country"] == country_name]

#     if len(df) == 0:
#         print("No samples found for country:", country_name)
#         return

#     # random selection
#     samples = df.sample(n=min(num_samples, len(df)))

#     # plot setup
#     fig, axes = plt.subplots(2, num_samples, figsize=(4*num_samples, 8))

#     if num_samples == 1:
#         axes = [[axes[0]], [axes[1]]]

#     for i, (_, row) in enumerate(samples.iterrows()):

#         ground_path = os.path.join(data_folder, row["gnd_image_path"])
#         sat_path = os.path.join(data_folder, row["sat_image_path"])
#         image_id = row["id"]

#         # load images
#         ground_img = Image.open(ground_path)
#         sat_img = Image.open(sat_path)

#         # display ground image (top)
#         axes[0][i].imshow(ground_img)
#         axes[0][i].set_title(f"ID: {image_id}")
#         axes[0][i].axis("off")

#         # display satellite image (bottom)
#         axes[1][i].imshow(sat_img)
#         axes[1][i].axis("off")

#     axes[0][0].set_ylabel("Ground", fontsize=14)
#     axes[1][0].set_ylabel("Satellite", fontsize=14)

#     plt.tight_layout()
#     plt.show()


# # example usage
# show_pairs(
#     csv_file="/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test.csv",
#     data_folder="/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k",
#     country_name="BR",
#     num_samples=5
# )



import pandas as pd
import random
import os
import matplotlib.pyplot as plt
from PIL import Image

def show_pairs(csv_file, data_folder, country_name, num_samples, img_size=(256,256), save_fig=False):

    # read csv
    df = pd.read_csv(csv_file)

    # filter by country
    df = df[df["unique_country"] == country_name]

    if len(df) == 0:
        print("No samples found for country:", country_name)
        return

    # randomly sample rows
    samples = df.sample(n=min(num_samples, len(df)))

    # create figure
    fig, axes = plt.subplots(2, num_samples, figsize=(4*num_samples, 8))

    if num_samples == 1:
        axes = [[axes[0]], [axes[1]]]

    for i, (_, row) in enumerate(samples.iterrows()):

        ground_path = os.path.join(data_folder, row["gnd_image_path"])
        sat_path = os.path.join(data_folder, row["sat_image_path"])
        image_id = row["id"]

        # load images
        ground_img = Image.open(ground_path).convert("RGB")
        sat_img = Image.open(sat_path).convert("RGB")

        # resize images to same size
        ground_img = ground_img.resize(img_size)
        sat_img = sat_img.resize(img_size)

        # top row: ground
        axes[0][i].imshow(ground_img)
        axes[0][i].set_title(f"ID: {image_id}", fontsize=10)
        axes[0][i].axis("off")

        # bottom row: satellite
        axes[1][i].imshow(sat_img)
        axes[1][i].axis("off")

    # row labels
    axes[0][0].set_ylabel("Ground", fontsize=14)
    axes[1][0].set_ylabel("Satellite", fontsize=14)

    # main title (country name)
    fig.suptitle(f"Country: {country_name}", fontsize=18)

    # plt.tight_layout()
    plt.subplots_adjust( left=0.03, right=0.97, top=0.88, bottom=0.05, wspace=0.01, hspace=0.01 )
    # plt.subplots_adjust(top=0.9)
    if save_fig: plt.savefig(f"fig/country_example/{country_name}.png")
    plt.show()


# example usage
show_pairs(
    # csv_file="/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test.csv",
    csv_file="csv/subset_all_final.csv",
    data_folder="/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k",
    country_name="ZA",
    num_samples=5,
    save_fig = True
)




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

csv_file = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test.csv"

# ======== Example Setup ========
# Read your CSV (or create DataFrame)
df = pd.read_csv(csv_file)
column_name = "unique_country"  # change this to your target column
# ================================


# Step 2: Count unique values and get top 10
value_counts = df[column_name].value_counts()
value_counts.to_csv('metadata/all_countries.csv')

# Step 3: Plot a bar chart
# plt.figure(figsize=(10, 6))
# value_counts.plot(kind='bar')
# plt.title(f"Top 10 Most Frequent Values in '{column_name}'")
# plt.xlabel(column_name)
# plt.ylabel("Count")
# plt.xticks(rotation=45, ha='right')
# plt.tight_layout()
# plt.show()

# filtering rows based on country codes and then sampling the same number of rows for each file.

In [ ]:
# filtering rows based on country codes and then sampling the same number of rows for each file.


# import pandas as pd

# df = pd.read_csv("/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test.csv")

# df = df[df["unique_country"].isin(["TZ"])]

# # df = df.sample(frac=0.15, random_state=42).reset_index(drop=True) 

# df.to_csv("csv/combine_countries/test_tz.csv", index=False)
# df.shape


import pandas as pd

def create_country_subsets(csv_path, countries, rows_per_file, output_prefix="subset", seed=42):
    
    df = pd.read_csv(csv_path)

    # keep only selected countries
    df = df[df['unique_country'].isin(countries)]

    # split dataframe by country
    country_dfs = {c: df[df['unique_country'] == c] for c in countries}

    # check available rows
    for c in countries:
        if len(country_dfs[c]) == 0:
            raise ValueError(f"No rows found for country {c}")

    for i in range(5):

        selected_countries = countries[:5-i]

        combined = pd.concat([country_dfs[c] for c in selected_countries])

        # sample rows
        if rows_per_file <= len(combined):
            subset = combined.sample(rows_per_file, random_state=seed)
        else:
            subset = combined.sample(rows_per_file, replace=True, random_state=seed)

        filename = f"csv/combine_countries/{output_prefix}_{5-i}_countries.csv"
        subset.to_csv(filename, index=False)

        print(f"{filename} created with {rows_per_file} rows using countries: {selected_countries}")


# Example usage
csv_path = "/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test_5k.csv"
countries = ['EC','RU','PE','BO','US']  # input 5 countries
rows_per_file = 1200                   # desired rows per output file

create_country_subsets(csv_path, countries, rows_per_file)

# Qualitative analysis


In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from PIL import Image
import random

def show_retrieval_row(csv_file, data_folder, gt_row, other_rows, img_size=(256,256)):

    df = pd.read_csv(csv_file)

    if len(other_rows) != 10:
        raise ValueError("Provide exactly 10 satellite row indices")

    # Ground truth row
    gt_data = df.iloc[gt_row]

    ground_path = os.path.join(data_folder, gt_data["gnd_image_path"])
    sat_gt_path = os.path.join(data_folder, gt_data["sat_image_path"])

    ground_img = Image.open(ground_path).convert("RGB").resize(img_size)
    sat_gt_img = Image.open(sat_gt_path).convert("RGB").resize(img_size)

    # create figure (12 images total)
    fig, axes = plt.subplots(1, 12, figsize=(24,4))

    # Ground image
    axes[0].imshow(ground_img)
    # axes[0].set_title("Ground")
    axes[0].axis("off")

    # GT satellite
    axes[1].imshow(sat_gt_img)
    # axes[1].set_title("GT Sat")
    axes[1].axis("off")

    # other satellite images
    for i, r in enumerate(other_rows):

        sat_path = os.path.join(data_folder, df.iloc[r]["sat_image_path"])
        sat_img = Image.open(sat_path).convert("RGB").resize(img_size)

        axes[i+2].imshow(sat_img)
        # axes[i+2].set_title(f"Sat {i+1}")
        axes[i+2].axis("off")

    plt.subplots_adjust(wspace=0.02)
    # plt.tight_layout()
    # plt.savefig(f"fig/top1/{gt_row}.png")
    plt.show()



gt_row = 332
other_rows = [random.randint(gt_row+5, 2000) for _ in range(10)]
other_rows[0]=gt_row
print(other_rows)
# Example usage
show_retrieval_row(
    csv_file="/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/test_5k.csv",
    data_folder="/home/fahimul/Documents/Research/Proj_worldDataset/datasets/osv500k/",
    gt_row=gt_row,
    other_rows=other_rows
)



## Rural vs Urban distribution

In [ ]:
import pandas as pd
import osmnx as ox
import seaborn as sns
import matplotlib.pyplot as plt


# -----------------------------
# Load dataset
# -----------------------------

csv_file = "datasets/osv500k/test_5k.csv"

# ======== Example Setup ========
# Read your CSV (or create DataFrame)
df = pd.read_csv(csv_file) #unique_country


# -----------------------------
# Downsample specific countries
# -----------------------------
def reduce_country(df, code, remove_percent):
    country_df = df[df["unique_country"] == code]
    keep_fraction = 1 - remove_percent
    kept = country_df.sample(frac=keep_fraction, random_state=42)
    df_other = df[df["unique_country"] != code]
    return pd.concat([df_other, kept])

# remove 80% of Brazil
df = reduce_country(df, "BR", 0.90)

# remove 40% of Argentina
df = reduce_country(df, "AR", 0.90)

# remove 40% of Columbia
df = reduce_country(df, "CO", 0.90)

df['idx'] = range(len(df))
# -----------------------------
# -----------------------------
def classify_area(lat, lon, idx):
    
    print(f'{idx}=>lat:{lat}, long:{lon}')
    try:
        tags = {"landuse": True}
        gdf = ox.features_from_point((lat, lon), tags=tags, dist=1000)

        urban_tags = ["residential", "commercial", "industrial"]

        if any(gdf["landuse"].isin(urban_tags)):
            return "Urban"
        else:
            return "Rural"

    except:
        return "Unknown"


print('start func')
df["area_type"] = df.apply(
    lambda row: classify_area(row["latitude"], row["longitude"], row['idx']),
    axis=1
)
print('end func')


# remove unknown
df = df[df["area_type"] != "Unknown"]

counts = df["area_type"].value_counts()

# plot
sns.set_theme(style="whitegrid", font_scale=1.5)

plt.figure(figsize=(7,7))

plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%"
)

plt.title("Urban vs Rural Distribution")
plt.axis("equal")

plt.show()

## Climate zone coverage

In [ ]:
import pandas as pd
import rasterio

# load dataset
df = pd.read_csv("datasets/osv500k/test.csv")

# -----------------------------
# Downsample specific countries
# -----------------------------
def reduce_country(df, code, remove_percent):
    country_df = df[df["unique_country"] == code]
    keep_fraction = 1 - remove_percent
    kept = country_df.sample(frac=keep_fraction, random_state=42)
    df_other = df[df["unique_country"] != code]
    return pd.concat([df_other, kept])

# remove 80% of Brazil
df = reduce_country(df, "BR", 0.90)

# remove 40% of Argentina
df = reduce_country(df, "AR", 0.90)

# remove 40% of Columbia
df = reduce_country(df, "CO", 0.90)



print(f'total samples: {df.shape}')

# open climate raster
climate = rasterio.open("metadata/Beck_KG_V1/Beck_KG_V1_present_0p0083.tif")

def get_climate(lat, lon):
    row, col = climate.index(lon, lat)
    value = climate.read(1)[row, col]
    return value

df["climate_zone"] = df.apply(
    lambda x: get_climate(x["latitude"], x["longitude"]),
    axis=1
)

In [ ]:
climate_map = {
    1: "Tropical",
    2: "Arid",
    3: "Temperate",
    4: "Cold",
    5: "Polar"
}

df["climate_zone"] = df["climate_zone"].map(climate_map)

In [ ]:
# Bar Chart
# import seaborn as sns
# import matplotlib.pyplot as plt

# sns.set_theme(style="whitegrid")

# counts = df["climate_zone"].value_counts()

# plt.figure(figsize=(8,5))
# sns.barplot(x=counts.index, y=counts.values)

# plt.xlabel("Climate Zone")
# plt.ylabel("Number of Samples")
# plt.title("Climate Zone Distribution")

# plt.show()
# **********************************************
# PI chart
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Example dataframe (after assigning climate zones)
# df["climate_zone"] should already exist
# Example values: Tropical, Arid, Temperate, Cold, Polar

# Count samples per climate zone
counts = df["climate_zone"].value_counts()

# Seaborn style
sns.set_theme(style="whitegrid", font_scale=1.4)

# Color palette
colors = sns.color_palette("pastel", len(counts))

plt.figure(figsize=(7,7))

plt.pie(
    counts,
    labels=counts.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=colors
)

# plt.title("Climate Zone Distribution")
plt.axis("equal")  # keeps the pie circular

plt.show()